# 02 - Preprocessing

In [23]:
import pandas as pd

In [24]:
# Load data 
months = ["202512", "202601", "202602", "202603", "202604", "202605"]
files = [f"california/CRMLSSold{m}.csv" for m in months]

df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
df = df[(df["PropertyType"] == "Residential") & (df["PropertySubType"] == "SingleFamilyResidence")]
df.shape

(61727, 78)

## Select columns

In [25]:
num_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt",
            "GarageSpaces", "ParkingTotal", "AssociationFee", "Latitude", "Longitude"]
bool_cols = ["PoolPrivateYN", "ViewYN", "FireplaceYN", "NewConstructionYN", "AttachedGarageYN"]
cat_cols = ["CountyOrParish"]

# Drop listings that show up in more than one file
df = df.drop_duplicates("ListingKey")
df["CloseMonth"] = pd.to_datetime(df["CloseDate"]).dt.strftime("%Y%m")
df = df[["ClosePrice", "CloseMonth"] + num_cols + bool_cols + cat_cols]
df.isna().sum()

ClosePrice                   0
CloseMonth                   0
LivingArea                  30
BedroomsTotal                0
BathroomsTotalInteger        1
LotSizeSquareFeet         1081
YearBuilt                   36
GarageSpaces              2353
ParkingTotal                 0
AssociationFee           17614
Latitude                     9
Longitude                    9
PoolPrivateYN             4833
ViewYN                    5295
FireplaceYN                 58
NewConstructionYN         4685
AttachedGarageYN          7521
CountyOrParish               0
dtype: int64

## Missing values

In [26]:
# Few rows are missing these, so drop them
df = df.dropna(subset=["LivingArea", "BathroomsTotalInteger", "LotSizeSquareFeet", "YearBuilt", "Latitude"])

# No fee listed means no HOA
df["AssociationFee"] = df["AssociationFee"].fillna(0)

# Fill the amount of garage spaces with the median amount of garage spaces
df["GarageSpaces"] = df["GarageSpaces"].fillna(df["GarageSpaces"].median())

# Treat missing Yes/No as No
df[bool_cols] = df[bool_cols].eq(True).astype(int)

df.isna().sum()

ClosePrice               0
CloseMonth               0
LivingArea               0
BedroomsTotal            0
BathroomsTotalInteger    0
LotSizeSquareFeet        0
YearBuilt                0
GarageSpaces             0
ParkingTotal             0
AssociationFee           0
Latitude                 0
Longitude                0
PoolPrivateYN            0
ViewYN                   0
FireplaceYN              0
NewConstructionYN        0
AttachedGarageYN         0
CountyOrParish           0
dtype: int64

## Encoding

In [27]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
df.shape

(60542, 74)

In [28]:
# Save the data to csv
df.to_csv("cleaned.csv", index=False)

## Train/test split

In [29]:
# Test on the most recent month, train on the X months before it
X = 4
train = df[df["CloseMonth"].isin(months[-1 - X:-1])].copy()
test = df[df["CloseMonth"] == months[-1]].copy()
train.shape, test.shape

((38458, 74), (11798, 74))

In [20]:
# Normalize numeric features using the training set's mean and std
mean, std = train[num_cols].mean(), train[num_cols].std()
train[num_cols] = (train[num_cols] - mean) / std
test[num_cols] = (test[num_cols] - mean) / std
train[num_cols].describe()

,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,GarageSpaces,ParkingTotal,AssociationFee,Latitude,Longitude
count,3.845800e+04,3.845800e+04,3.845800e+04,3.845800e+04,3.845800e+04,3.845800e+04,3.845800e+04,3.845800e+04,3.845800e+04,3.845800e+04
mean,8.277163e-17,-1.980607e-16,-1.241574e-16,1.478065e-18,3.043336e-15,-2.217097e-17,-3.695162e-18,-1.478065e-18,5.986163e-16,2.233356e-15
std,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,-1.981916e+00,-3.605574e+00,-2.331747e+00,-2.018655e-02,-6.333511e+00,-1.119317e+00,-8.500813e-01,-2.486508e-01,-3.220983e+01,-1.519731e+00
25%,-6.437941e-01,-5.110588e-01,-5.675116e-01,-1.991853e-02,-7.396485e-01,2.925393e-03,-4.414994e-02,-2.486508e-01,-5.362012e-01,-1.440305e-01
50%,-2.247277e-01,-5.110588e-01,-5.675116e-01,-1.984231e-02,1.822958e-02,2.925393e-03,-4.414994e-02,-2.486508e-01,-3.574565e-01,1.550611e-01
75%,3.889397e-01,5.204463e-01,3.146063e-01,-1.969178e-02,8.121971e-01,2.925393e-03,6.240274e-04,-1.058554e-01,-1.233663e-02,3.752153e-01
max,1.677877e+01,1.908754e+01,1.354638e+01,8.978414e+01,1.786612e+00,1.672170e+02,1.754250e+02,6.267854e+01,4.002534e+00,6.834543e+01
